# 02 · Instrumentar de verdad

**Módulo 1 · Trazas** — *tiempo estimado: 75 minutos* — *consumo: 0 trazas*

El notebook anterior abrió una traza por dentro con funciones de juguete. Este va de
poner eso en una aplicación real, que es donde aparecen los problemas.

Al terminar sabrás:

1. **Qué se instrumenta solo** —y es mucho— y qué no.
2. Instrumentar el SDK de un proveedor a pelo con `wrap_openai`, y por qué eso importa
   aunque uses LangChain.
3. Las **variables de entorno**: cuántas hay de verdad, las que se usan, y **dos trampas**
   que hacen perder tardes enteras.
4. Poner nombres, etiquetas y metadatos **donde sirven**.
5. Mandar trazas a proyectos distintos según el contexto.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                            if (p / "utils" / "curso.py").exists())))

from utils.curso import init, online, cliente, traza_local, separador
from langsmith import traceable
from langsmith.run_helpers import get_current_run_tree

init(silencioso=True)
print("listo · sin clave, todo lo de abajo se ejecuta igual")

## 1. Lo que se instrumenta solo

Esta es la razón de peso para elegir LangSmith si ya usas LangChain o LangGraph: **no
tienes que hacer nada**. Cualquier `Runnable` —una cadena, un grafo, un modelo, un
recuperador— emite eventos a través del sistema de *callbacks*, y el tracer los recoge.

Vamos a comprobarlo con un grafo de LangGraph de los del otro curso.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel

# Un modelo falso, para que este notebook no necesite clave ni gaste dinero.
modelo = FakeMessagesListChatModel(responses=[
    AIMessage("facturacion"),
    AIMessage("Te devolvemos el cargo duplicado en 5 días hábiles."),
])

class Estado(TypedDict):
    messages: list
    categoria: str

def clasificar(estado: Estado) -> dict:
    respuesta = modelo.invoke(estado["messages"])
    return {"categoria": respuesta.text}

def responder(estado: Estado) -> dict:
    respuesta = modelo.invoke(estado["messages"])
    return {"messages": estado["messages"] + [respuesta]}

grafo = StateGraph(Estado)
grafo.add_node("clasificar", clasificar)
grafo.add_node("responder", responder)
grafo.add_edge(START, "clasificar")
grafo.add_edge("clasificar", "responder")
grafo.add_edge("responder", END)
app = grafo.compile()

with traza_local() as t:
    resultado = app.invoke({"messages": [HumanMessage("Me habéis cobrado dos veces")],
                            "categoria": ""})

print("categoría:", resultado["categoria"])
print()
t.dibujar()

Cinco runs, la jerarquía correcta, los nodos con su nombre y las llamadas al modelo
marcadas como `llm` — y **no hemos escrito ni un decorador**. Eso es lo que se paga
cuando se paga LangSmith.

Lo que se instrumenta solo:

| Pieza | Se traza | `run_type` |
|---|---|---|
| Un grafo de LangGraph | Sí, con un run por nodo y por superpaso | `chain` |
| Un `Runnable` de LangChain (LCEL) | Sí, con un run por paso | `chain` |
| Un modelo de chat | Sí, **con tokens y coste** | `llm` |
| Una `@tool` de LangChain | Sí | `tool` |
| Un recuperador | Sí, con los documentos | `retriever` |

## 2. Lo que NO se instrumenta solo

Todo lo demás. Y «todo lo demás» en una aplicación real es la mitad del código:

- Tus funciones de Python normales: validar la entrada, decidir a qué agente enrutar,
  post-procesar la salida.
- Llamadas al SDK del proveedor **sin pasar por LangChain**.
- Consultas a tu base de datos, a tu API interna, a un servicio de terceros.
- El código asíncrono que lanzas tú.

En la traza de arriba, si `clasificar` hubiera hecho una consulta a Postgres que tarda
800 ms, **no la verías**. Verías un nodo `clasificar` que tarda 850 ms sin explicación,
y te pasarías la tarde sospechando del modelo.

La solución es `@traceable`, que ya conoces. Lo que aporta este apartado es **dónde
ponerlo**, y la regla es corta:

> Decora lo que **pueda ser lento o pueda fallar**. Todo lo que quieras poder acusar.

In [ ]:
import time

@traceable(run_type="tool", name="consulta_postgres")
def buscar_historial(cliente_id: str) -> list[dict]:
    """Sin este decorador, sus 50 ms se los come el nodo que la llama."""
    time.sleep(0.05)
    return [{"ticket": "TCK-0001", "estado": "cerrado"}]

def clasificar_instrumentado(estado: Estado) -> dict:
    buscar_historial("acme")
    respuesta = modelo.invoke(estado["messages"])
    return {"categoria": respuesta.text}

grafo2 = StateGraph(Estado)
grafo2.add_node("clasificar", clasificar_instrumentado)
grafo2.add_edge(START, "clasificar")
grafo2.add_edge("clasificar", END)
app2 = grafo2.compile()

modelo.i = 0                      # el modelo falso lleva un índice interno
with traza_local() as t:
    app2.invoke({"messages": [HumanMessage("Me habéis cobrado dos veces")], "categoria": ""})

def ms(r):
    return (r.end_time - r.start_time).total_seconds() * 1000

for p, r in t.recorrer():
    print(f"{'  ' * p}{r.name:<20} [{r.run_type:<5}] {ms(r):6.1f} ms")

Ahora la consulta a la base de datos aparece con su nombre y con su tiempo, y se ve de
un vistazo que el grueso del nodo no es el modelo.

## 3. El SDK del proveedor a pelo: `wrap_openai`

Caso muy frecuente: usas LangGraph para orquestar, pero **una** llamada la haces con el
cliente de OpenAI directamente, porque necesitas un parámetro que el envoltorio no
expone. Esa llamada no se traza. Y como es una llamada al modelo, además pierdes los
tokens y el coste, que es justo lo que querías medir.

`langsmith.wrappers` lo resuelve envolviendo el cliente:

```python
from openai import OpenAI
from langsmith.wrappers import wrap_openai

cliente_openai = wrap_openai(OpenAI())      # <- lo único que cambia

respuesta = cliente_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "hola"}],
)
```

A partir de ahí cada llamada aparece como un run `llm`, con sus tokens y su coste, y se
anida donde toque si está dentro de otro run.

Hay envoltorios para los tres grandes:

| Función | Para |
|---|---|
| `wrap_openai` | `openai.OpenAI` y `AsyncOpenAI` (y todo lo compatible con su API) |
| `wrap_anthropic` | `anthropic.Anthropic` |
| `wrap_gemini` | El SDK de Google |

In [ ]:
# Comprobamos que existen y qué firma tienen, sin llamar a nadie.
import inspect
from langsmith import wrappers

for nombre in ("wrap_openai", "wrap_anthropic", "wrap_gemini"):
    fn = getattr(wrappers, nombre)
    parametros = list(inspect.signature(fn).parameters)
    print(f"{nombre:<16} {parametros}")

Fíjate en el parámetro `tracing_extra`: ahí van las etiquetas y metadatos que quieras
para *todas* las llamadas de ese cliente. Es el sitio natural para marcar «este cliente
es el del modelo barato» y poder separar el gasto después.

> **Regla práctica.** Si el `run_type` de una llamada al modelo no es `llm`, no se
> contabiliza. Un `@traceable(run_type="chain")` alrededor de una llamada a OpenAI
> traza la llamada y **no** cuenta los tokens. Usa `wrap_openai`, no un decorador.

## 4. Las variables de entorno, con sus dos trampas

La documentación cubre cinco o seis. Vamos a contar las que el SDK **de verdad lee**,
mirándolo por dentro en lugar de creerme a mí.

In [ ]:
import re, pathlib
import langsmith

raiz_sdk = pathlib.Path(langsmith.__file__).parent
literales, dinamicas = set(), set()

for fichero in raiz_sdk.rglob("*.py"):
    texto = fichero.read_text(encoding="utf-8", errors="ignore")
    # 1. Las escritas tal cual: os.environ.get("LANGSMITH_...")
    literales |= set(re.findall(
        r'(?:os\.environ\.get|os\.getenv|os\.environ\[)\(?\s*["\']'
        r'(LANGSMITH_[A-Z0-9_]+|LANGCHAIN_[A-Z0-9_]+)["\']', texto))
    # 2. Las construidas: get_env_var("TRACING") -> LANGSMITH_TRACING o LANGCHAIN_TRACING
    for patron in (r'get_env_var\(\s*["\']([a-zA-Z0-9_]+)["\']',
                   r'get_bool_env_var\(\s*["\']([a-zA-Z0-9_]+)["\']',
                   r'get_str_env_var\(\s*["\']([a-zA-Z0-9_]+)["\']',
                   r'is_env_var_truish\(\s*["\']([a-zA-Z0-9_]+)["\']'):
        dinamicas |= set(re.findall(patron, texto))

logicos = {v.split("_", 1)[1] for v in literales} | {d.upper() for d in dinamicas}
grafias = literales | {f"{ns}_{n}" for n in logicos for ns in ("LANGSMITH", "LANGCHAIN")}

print(f"nombres lógicos distintos : {len(logicos)}")
print(f"grafías que el SDK acepta : {len(grafias)}   (cada una con los dos prefijos)")
print(f"\nlas documentadas de forma prominente son unas 6.")

Treinta y siete conceptos, setenta y cuatro grafías. No hace falta saberlas, pero sí
saber que existen, porque **cuando algo no funciona la respuesta suele ser una de
ellas**. Estas son las que resuelven problemas reales:

| Variable | Para qué | Cuándo la vas a necesitar |
|---|---|---|
| `LANGSMITH_TRACING` | El interruptor | Siempre |
| `LANGSMITH_API_KEY` | La clave | Siempre |
| `LANGSMITH_PROJECT` | Dónde caen las trazas | Siempre |
| `LANGSMITH_ENDPOINT` | Región | Si estás en Europa |
| `LANGSMITH_TRACING_SAMPLING_RATE` | Traza solo una fracción (0 a 1) | Cuando produzcas más de lo que cabe en tu plan (nb 03) |
| `LANGSMITH_HIDE_INPUTS` / `_OUTPUTS` / `_METADATA` | Manda la forma, no el contenido | Cuando los datos no puedan salir (nb 05) |
| `LANGSMITH_TEST_CACHE` | Caché de respuestas del modelo en pruebas | Para tener CI con LLM determinista y gratis (nb 10) |
| `LANGSMITH_TEST_TRACKING` | Apaga el registro en las pruebas | Para que la CI no te gaste la cuota |
| `LANGSMITH_FAILED_TRACES_DIR` | Dónde guardar lo que no se pudo enviar | Cuando pierdas trazas (nb 03) |
| `LANGSMITH_FAILED_TRACES_MAX_MB` | Cuánto guardar antes de tirarlo | Lo mismo |
| `LANGSMITH_RUNS_ENDPOINTS` | Enviar a varios destinos a la vez | Migraciones y dobles escrituras |

### Trampa 1 · Los dos prefijos, y el que gana

Todos los nombres funcionan con `LANGSMITH_` y con `LANGCHAIN_`. Los segundos son los
antiguos y siguen ahí por compatibilidad. El SDK los busca **en ese orden**:

```python
names = [f"{namespace}_{name}" for namespace in ("LANGSMITH", "LANGCHAIN")]
for name in names:
    value = os.environ.get(name)
    if value is not None and value.strip() != "":
        return value
```

O sea: **`LANGSMITH_` gana siempre**, aunque `LANGCHAIN_` esté puesta después, aunque
esté en el `.env` y la otra en la sesión.

El síntoma cuando esto muerde: heredas un proyecto con `LANGCHAIN_PROJECT=produccion`
en el despliegue, tú añades `LANGSMITH_PROJECT=mi-prueba` para depurar, se te olvida
quitarlo, y **todo el tráfico de producción se va a `mi-prueba`** sin un solo aviso.

**Recomendación: deja solo las `LANGSMITH_`.** Y si heredas un proyecto, lo primero es
comprobar si hay de las dos.

In [ ]:
def revisar_prefijos_duplicados(entorno: dict) -> list[str]:
    """Encuentra las variables definidas con los dos prefijos, que es donde muerde."""
    avisos = []
    for clave, valor in entorno.items():
        if not clave.startswith("LANGCHAIN_"):
            continue
        equivalente = "LANGSMITH_" + clave.removeprefix("LANGCHAIN_")
        if equivalente in entorno:
            gana = "LANGSMITH" if entorno[equivalente].strip() else "LANGCHAIN"
            avisos.append(f"{clave}={valor!r} y {equivalente}={entorno[equivalente]!r} -> gana {gana}")
    return avisos

entorno_heredado = {
    "LANGCHAIN_PROJECT": "produccion",
    "LANGSMITH_PROJECT": "mi-prueba-de-ayer",
    "LANGCHAIN_TRACING_V2": "true",
    "OPENAI_API_KEY": "sk-...",
}

for aviso in revisar_prefijos_duplicados(entorno_heredado):
    print("[AVISO]", aviso)

### Trampa 2 · La caché que hace que tu cambio no sirva de nada

Esta es más sutil, y es la que provoca el «he puesto la variable y no pasa nada» que
inunda los foros.

`get_env_var` está decorada con `@functools.lru_cache(maxsize=100)`. La primera vez que
el SDK lee una variable, **se queda con el valor**. Cambiarla después en `os.environ`
no tiene ningún efecto.

En un script no se nota, porque las lees al arrancar. **En un notebook se nota mucho**,
porque el kernel vive horas y tú cambias cosas por el camino.

In [ ]:
import os
from langsmith import utils as lu

lu.get_env_var.cache_clear()          # partimos de limpio para la demostración
os.environ["LANGSMITH_PROJECT"] = "proyecto-a"
print("1. el SDK lee por primera vez  ->", lu.get_env_var("PROJECT"))

os.environ["LANGSMITH_PROJECT"] = "proyecto-b"
print("\n2. cambias la variable en una celda")
print("   os.environ dice             ->", os.environ["LANGSMITH_PROJECT"])
print("   el SDK dice                 ->", lu.get_env_var("PROJECT"), "  <-- SIGUE EN EL VIEJO")

lu.get_env_var.cache_clear()
print("\n3. tras get_env_var.cache_clear()")
print("   el SDK dice                 ->", lu.get_env_var("PROJECT"))

Dos formas de no pisarla:

- **En un notebook:** reinicia el kernel después de tocar el `.env`. Es lo aburrido y lo
  fiable.
- **Si necesitas cambiar en caliente:** no uses variables de entorno. Usa
  `tracing_context(project_name=...)` o pásale el `project_name` a la llamada, que se
  leen cada vez. Es lo que hace el apartado siguiente.

## 5. Nombres, etiquetas y metadatos donde sirven

Ya sabes del notebook 01 que se heredan hacia abajo. Aquí va **qué poner**, que es la
parte que no se cuenta en ningún sitio.

El criterio: los metadatos existen para poder **filtrar y agrupar después**. La pregunta
que hay que hacerse al instrumentar no es «¿qué información tengo?» sino **«¿qué voy a
querer preguntarle a esto dentro de tres semanas, a las tres de la mañana?»**.

Las preguntas de las tres de la mañana son casi siempre las mismas cinco:

| Pregunta | Lo que hace falta |
|---|---|
| «¿A este cliente le funciona?» | `metadata["cliente"]` |
| «¿La versión nueva es peor?» | `metadata["version"]` o una etiqueta |
| «¿Es solo este tipo de ticket?» | `metadata["categoria"]` |
| «¿Cuál es la conversación entera?» | `metadata["session_id"]` (notebook 04) |
| «¿Esto era producción o una prueba?» | una etiqueta `produccion` / `desarrollo` |

Y todo eso se pone **en el punto de entrada**, una sola vez, porque baja solo.

In [ ]:
@traceable(run_type="tool")
def politica_de_reembolsos(categoria: str) -> str:
    return "art. 14"

@traceable(run_type="llm")
def generar(pregunta: str, politica: str) -> str:
    return "Te lo devolvemos en 5 días."

@traceable(run_type="chain", name="atender_ticket")
def atender_ticket(ticket: dict) -> str:
    politica = politica_de_reembolsos(ticket["categoria"])
    return generar(ticket["mensaje"], politica)


def punto_de_entrada(ticket: dict) -> str:
    """El ÚNICO sitio del código que decide qué metadatos lleva la traza."""
    return atender_ticket(ticket, langsmith_extra={
        "metadata": {
            "cliente": ticket["cliente"],
            "plan": ticket["plan"],
            "categoria": ticket["categoria"],
            "version_prompt": "v3",
        },
        "tags": ["produccion", f"plan-{ticket['plan']}"],
        # La clave es `name`, no `run_name`. Ver el aviso de debajo.
        "name": f"ticket-{ticket['id']}",         # el nombre que verás en la lista
    })

with traza_local() as t:
    punto_de_entrada({"id": "TCK-0001", "cliente": "acme", "plan": "free",
                      "categoria": "facturacion", "mensaje": "cobro duplicado"})

for p, r in t.recorrer():
    meta = {k: v for k, v in r.extra["metadata"].items() if k != "ls_method"}
    print(f"{'  ' * p}{r.name}")
    print(f"{'  ' * p}   tags={r.tags}")
    print(f"{'  ' * p}   meta={meta}")

Fíjate en el nombre de la raíz: `ticket-TCK-0001` y no `atender_ticket`. En una lista de
trazas eso es la diferencia entre mil filas idénticas y mil filas que puedes leer.

> **Trampa 3, y es de las que cuestan una tarde.** La clave es `name`. Si escribes
> `run_name` —que es como se llama en `@traceable(run_name=...)` en algunas versiones y
> lo que dicta la intuición— **no pasa nada de nada**: no hay error, no hay aviso, y la
> traza se llama como la función. `langsmith_extra` **ignora en silencio las claves que
> no conoce**. Estas son las que sí conoce:

In [ ]:
from langsmith.run_helpers import LangSmithExtra

print("claves válidas de langsmith_extra:")
for clave in LangSmithExtra.__annotations__:
    if not clave.startswith("_"):
        print("  ", clave)

print()
@traceable
def ejemplo(x):
    return x

for clave in ("name", "run_name"):
    with traza_local() as t_nombre:
        ejemplo(1, langsmith_extra={clave: "MI-NOMBRE"})
    print(f"  langsmith_extra={{{clave!r}: 'MI-NOMBRE'}} -> la traza se llama "
          f"{t_nombre.principales[0].name!r}")

Es el mismo patrón que ya has visto dos veces en este módulo: **el SDK prefiere seguir
funcionando a avisarte**. Con el `run_type` equivocado, con la caché de variables, y
aquí. Ninguna de las tres da un error; las tres dan un resultado silenciosamente
distinto del que esperas. Tenerlo presente es la mitad de saber depurar esto.

## 6. Proyectos distintos según el contexto

Un proyecto es un cajón. Lo típico es querer tres: producción, desarrollo y
experimentos. Hay tres formas de elegir, y **la de más abajo gana**:

1. `LANGSMITH_PROJECT` en el entorno — el valor por defecto.
2. `tracing_context(project_name=...)` — para un bloque.
3. `langsmith_extra={"project_name": ...}` — para una llamada.

Las dos últimas **no** pasan por la caché de la trampa 2, así que sirven para decidir en
caliente. Es como se separa el tráfico interno del de clientes sin desplegar nada.

In [ ]:
from langsmith.run_helpers import tracing_context

def enrutar_por_origen(ticket: dict) -> str:
    proyecto = "soporte-produccion" if ticket["origen"] == "cliente" else "soporte-pruebas"
    with tracing_context(project_name=proyecto):
        return atender_ticket(ticket)

# Sin servicio no se envía nada, pero la decisión sí se puede comprobar.
for origen in ("cliente", "equipo-interno"):
    proyecto = "soporte-produccion" if origen == "cliente" else "soporte-pruebas"
    print(f"  origen={origen:<15} -> proyecto={proyecto}")

In [ ]:
@online("Comprobar en qué proyecto han caído las trazas", trazas=0)
def _():
    for p in cliente().list_projects(limit=10):
        print(f"  {p.name}")

## 7. Ejercicios

### Ejercicio 1 — El agujero de 800 ms

Aquí tienes un nodo que tarda mucho y una traza que no explica por qué. Instrumenta lo
que haga falta para que la traza responda **dónde se va el tiempo**, y comprueba que la
respuesta sale de la traza y no de leer el código.

In [ ]:
def buscar_en_indice(consulta: str) -> list[str]:
    time.sleep(0.04)
    return ["doc-1", "doc-2"]

def reordenar(docs: list[str]) -> list[str]:
    time.sleep(0.02)
    return list(reversed(docs))

def nodo_recuperacion(estado: Estado) -> dict:
    docs = buscar_en_indice(estado["messages"][-1].text)
    docs = reordenar(docs)
    modelo.i = 0
    modelo.invoke(estado["messages"])
    return {"categoria": "listo"}

# Tu solución aquí: instrumenta y mide.

<details>
<summary><b>Solución</b></summary>

In [ ]:
@traceable(run_type="retriever", name="indice_vectorial")
def buscar_en_indice_trazado(consulta: str) -> list[str]:
    time.sleep(0.04)
    return ["doc-1", "doc-2"]

@traceable(run_type="tool", name="reordenador")
def reordenar_trazado(docs: list[str]) -> list[str]:
    time.sleep(0.02)
    return list(reversed(docs))

def nodo_recuperacion_trazado(estado: Estado) -> dict:
    docs = buscar_en_indice_trazado(estado["messages"][-1].text)
    docs = reordenar_trazado(docs)
    modelo.i = 0
    modelo.invoke(estado["messages"])
    return {"categoria": "listo"}

g = StateGraph(Estado)
g.add_node("recuperacion", nodo_recuperacion_trazado)
g.add_edge(START, "recuperacion")
g.add_edge("recuperacion", END)
app_ej = g.compile()

with traza_local() as t:
    app_ej.invoke({"messages": [HumanMessage("cobro duplicado")], "categoria": ""})

raiz = t.principales[0]
total = ms(raiz)
print(f"total: {total:.0f} ms\n")
for p, r in t.recorrer():
    print(f"{'  ' * p}{r.name:<20} {ms(r):6.1f} ms  ({100 * ms(r) / total:4.1f} %)")

Ahora la traza dice sola que el índice se lleva la mayor parte y que el modelo casi no
pesa. Sin los dos decoradores, el nodo `recuperacion` aparecía como una caja de 60 ms
sin nada dentro y la sospecha habría caído en el modelo, que es lo caro y lo lento *en
general* — pero no aquí.

Esa es toda la tesis de instrumentar: **que la traza responda sin que tengas que abrir
el código.**

</details>

### Ejercicio 2 — Un auditor de instrumentación

Escribe `auditar(traza, metadatos_obligatorios)` que revise una traza y avise de los
tres fallos de instrumentación de este notebook:

1. Runs cuyo nombre es el de la función (`atender_ticket` en vez de `ticket-TCK-0001`)
   — la traza será ilegible en una lista.
2. La raíz sin alguno de los metadatos obligatorios.
3. Runs `chain` cuyo nombre sugiera una llamada al modelo (`llm`, `modelo`, `gpt`,
   `chat`, `completion`) — el error del `run_type` que vacía el panel de coste.

In [ ]:
# Tu solución aquí.

<details>
<summary><b>Solución</b></summary>

In [ ]:
SOSPECHOSOS = ("llm", "modelo", "gpt", "chat", "completion", "claude")

def auditar(traza, metadatos_obligatorios=("cliente", "version_prompt")):
    avisos = []

    for raiz in traza.principales:
        meta = raiz.extra.get("metadata", {})
        faltan = [m for m in metadatos_obligatorios if m not in meta]
        if faltan:
            avisos.append(f"la raíz «{raiz.name}» no lleva {faltan} — "
                          "no podrás filtrar por eso dentro de tres semanas")

    for _, run in traza.recorrer():
        if run.run_type == "chain" and any(s in run.name.lower() for s in SOSPECHOSOS):
            avisos.append(f"«{run.name}» es chain y parece una llamada al modelo — "
                          "no contará tokens ni coste")

    # ¿La raíz conserva el nombre de la función? Si el nombre coincide con el de una
    # función del módulo, nadie le puso un `name` y en la lista serán filas iguales.
    for raiz in traza.principales:
        if raiz.name in globals() and callable(globals()[raiz.name]):
            avisos.append(f"la raíz se llama «{raiz.name}», el nombre de la función — "
                          "en una lista de mil trazas serán mil filas iguales")

    return avisos


@traceable(run_type="chain", name="llamar_al_modelo")     # mal el run_type
def llamar_al_modelo(p):
    return "r"

@traceable(run_type="chain")                               # sin metadatos ni run_name
def atender_ticket_mal(t):
    return llamar_al_modelo(t)

with traza_local() as mala:
    atender_ticket_mal("hola")

print("=== instrumentación descuidada ===")
for a in auditar(mala):
    print("  [AVISO]", a)

with traza_local() as buena:
    punto_de_entrada({"id": "TCK-0002", "cliente": "acme", "plan": "pro",
                      "categoria": "facturacion", "mensaje": "otro cobro"})

print("\n=== instrumentación del apartado 5 ===")
print("  ", auditar(buena) or "sin avisos")

Tres avisos y ninguno. Una función de treinta líneas que, puesta en la CI sobre una
traza de ejemplo, impide que la instrumentación se degrade con el tiempo — que es lo
que siempre pasa: se instrumenta bien el primer mes y luego cada función nueva entra
sin metadatos.

El módulo 4 lleva esta idea a producción: las mismas comprobaciones, pero como reglas
que corren sobre las trazas reales.

</details>

## 8. Resumen

- **LangChain y LangGraph se instrumentan solos**, con la jerarquía y los `run_type`
  correctos. Ese es el argumento de peso de la herramienta.
- **Nada más se instrumenta solo.** Decora lo que pueda ser lento o pueda fallar: si no,
  su tiempo se lo come el nodo que la llama y acusas al modelo sin motivo.
- Para el SDK del proveedor a pelo, `wrap_openai` y hermanos. Un `@traceable` alrededor
  **traza pero no cuenta tokens**; solo `run_type="llm"` cuenta.
- El SDK lee **37 variables lógicas y 74 grafías**. Dos trampas: `LANGSMITH_` gana
  siempre sobre `LANGCHAIN_`, y `get_env_var` está **cacheada**, así que cambiar una
  variable en caliente no hace nada hasta reiniciar el kernel.
- Los metadatos se ponen **en el punto de entrada**, una vez, y bajan solos. Elígelos
  pensando en las preguntas de las tres de la mañana, no en la información que tienes.
- El proyecto se decide por entorno, por bloque (`tracing_context`) o por llamada — y
  las dos últimas esquivan la caché.

**Siguiente:** [`03_trazas_que_se_pierden`](03_trazas_que_se_pierden.ipynb) — instrumentar
bien no sirve de nada si las trazas no llegan, y hay más formas de perderlas de las que
parece.